# QDF 이중 태스크 학습 → 3DGCL 정확도 기여 비교

## 전체 파이프라인

```
┌─────────────────────────────────────────────────────────────────────┐
│  [QM9] ──► QDF E-only 학습 ─────────────────────┐                  │
│  [QM9] ──► QDF E+V 이중 태스크 학습 ──────────────┼─► 체크포인트 저장 │
│                                                  │                  │
│  [ESOL MMFF 컨포머] + 각 체크포인트                │                  │
│        ──► QDF 추론 ──► 에너지 예측 CSV           │                  │
│        ──► aux .pt (분자별 에너지 타겟)            ◄─┘                │
│                                                                     │
│  [ESOL 그래프]                                                      │
│    + 컨포머 가중치(.pt)   ─► 3DGCL 대조 사전학습                     │
│    + aux.pt (E-only)           λ × MSE(임베딩, QDF_eonly 예측)      │
│    + aux.pt (E+V)    ─► 3DGCL 대조 사전학습                         │
│                                λ × MSE(임베딩, QDF_ev 예측)         │
│        ↓                                                            │
│   [미세조정] ──► ESOL RMSE ↓ (E+V가 E-only보다 낮을 것으로 예상)     │
└─────────────────────────────────────────────────────────────────────┘
```

## 비교 조건

| 조건 | QDF aux | 설명 |
|------|---------|------|
| **A: 기준선** | 없음 | 순수 GraphCL (no QDF) |
| **B: E-only** | E-only QDF 예측 | V(HK map) 없이 에너지만 학습한 QDF |
| **C: E+V** | E+V QDF 예측 | 이중 태스크(에너지 + 전자 포텐셜) 학습한 QDF |

> **가설**: E+V QDF는 물리적으로 더 일관된 에너지 표현을 학습하므로,
> 그 예측값을 auxiliary 신호로 사용하면 3DGCL이 더 좋은 분자 표현을 학습하고
> 최종 ESOL RMSE가 낮아진다.

## 0. 공통 설정

In [ ]:
import os, sys, csv, pickle, time, gc
import numpy as np
import torch
import torch.optim as optim
from pathlib import Path

# ── 경로 설정 ──────────────────────────────────────────────────────────
NB_DIR = Path.cwd()
if NB_DIR.name != 'bench':
    NB_DIR = Path('QuantumDeepField_molecule/bench').resolve()
QDF_ROOT  = NB_DIR.parent
REPO_ROOT = QDF_ROOT.parent

for _p in [str(QDF_ROOT / 'train'),
           str(REPO_ROOT),
           str(REPO_ROOT / 'examples' / 'sslgraph' / 'bench')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import train as qdf_train

# ── 디바이스 ────────────────────────────────────────────────────────────
if hasattr(torch, 'xpu') and torch.xpu.is_available():
    DEVICE = torch.device('xpu')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')

# ══════════════════════════════════════════════════════════════════════
#  QDF 설정 (QM9under7atoms — 빠른 데모용)
# ══════════════════════════════════════════════════════════════════════
QDF_DATASET    = 'QM9under7atoms_atomizationenergy_eV'
QDF_OPERATION  = 'sum'
BASIS_SET      = '6-31G'
RADIUS         = '0.75'
GRID_INTERVAL  = '0.3'
LOADER         = 'shard'

QDF_DIM              = 50
QDF_HIDDEN_HK        = 50
QDF_LAYER_FUNCTIONAL = 3
QDF_LAYER_HK         = 3
QDF_BATCH_SIZE       = 8
QDF_LR               = 1e-4
QDF_LR_DECAY         = 0.99
QDF_STEP_SIZE        = 1
QDF_N_EPOCHS         = 15    # ← 빠른 데모. 더 좋은 결과: 50+
NUM_WORKERS          = 0

# ══════════════════════════════════════════════════════════════════════
#  3DGCL 설정
# ══════════════════════════════════════════════════════════════════════
DGCL_DATASET    = 'esol'
DGCL_BATCH      = 400
QDF_AUX_LAMBDA  = 0.05   # 3DGCL aux 손실 가중치
WEIGHT_KT       = 0.5

# smoke 버짓: P_EPOCH=7, F_EPOCH=26, N_FOLDS=2
# two_hour 버짓: P_EPOCH=15, F_EPOCH=50, N_FOLDS=3
QUALITY_BUDGET  = 'smoke'   # 'smoke' | 'two_hour'

if QUALITY_BUDGET == 'smoke':
    P_EPOCH, F_EPOCH, N_FOLDS, N_TIMES = 7, 26, 2, 1
else:
    P_EPOCH, F_EPOCH, N_FOLDS, N_TIMES = 15, 50, 3, 1

SEED = 42
torch.manual_seed(SEED)

# 체크포인트 및 중간 파일 저장 경로
CKPT_DIR = NB_DIR / 'ensemble_ckpts'
CKPT_DIR.mkdir(exist_ok=True)
AUX_DIR  = NB_DIR / 'ensemble_aux'
AUX_DIR.mkdir(exist_ok=True)

CKPT_EONLY = CKPT_DIR / 'qdf_eonly.pt'
CKPT_EV    = CKPT_DIR / 'qdf_ev.pt'
CSV_EONLY  = AUX_DIR  / 'esol_qdf_mmff_preds_eonly.csv'
CSV_EV     = AUX_DIR  / 'esol_qdf_mmff_preds_ev.csv'
AUX_EONLY  = AUX_DIR  / 'esol_qdf_aux_eonly.pt'
AUX_EV     = AUX_DIR  / 'esol_qdf_aux_ev.pt'

# 컨포머 선택 가중치 (기존 잘 훈련된 E+V 모델 기준, 양 조건 공통)
WEIGHTS_PT = REPO_ROOT / 'dataset' / 'esol_mmff_weights_qdf_atomization.pt'
if not WEIGHTS_PT.exists():
    WEIGHTS_PT = REPO_ROOT / 'dataset' / 'esol_mmff_weights_boltzmann.pt'
print(f'MMFF weights: {WEIGHTS_PT.name}  exists={WEIGHTS_PT.exists()}')

print(f'\nQDF_N_EPOCHS={QDF_N_EPOCHS}, QUALITY_BUDGET={QUALITY_BUDGET}')
print(f'P_EPOCH={P_EPOCH}, F_EPOCH={F_EPOCH}, N_FOLDS={N_FOLDS}')

---
## 1. QDF 학습: E-only vs E+V 이중 태스크

동일한 아키텍처·초기값·데이터셋에서 두 가지 트레이너로 학습합니다:

- **E-only**: 에너지 손실만 최적화  
- **E+V**: 에너지 + HK map(전자 포텐셜) 이중 태스크

In [ ]:
# ── QM9 데이터 로더 ──────────────────────────────────────────────────
DIR_DATASET = QDF_ROOT / 'dataset' / QDF_DATASET
FIELD       = f'{BASIS_SET}_{RADIUS}sphere_{GRID_INTERVAL}grid'

def make_qdf_dataloaders():
    from dataset_shard import MyDatasetShard, default_shard_path
    ds_train = MyDatasetShard(default_shard_path(DIR_DATASET / f'train_{FIELD}'))
    ds_val   = MyDatasetShard(default_shard_path(DIR_DATASET / f'val_{FIELD}'))
    ds_test  = MyDatasetShard(default_shard_path(DIR_DATASET / f'test_{FIELD}'))
    dl_train = qdf_train.mydataloader(ds_train, QDF_BATCH_SIZE, NUM_WORKERS, shuffle=True)
    dl_val   = qdf_train.mydataloader(ds_val,   QDF_BATCH_SIZE, NUM_WORKERS)
    dl_test  = qdf_train.mydataloader(ds_test,  QDF_BATCH_SIZE, NUM_WORKERS)
    return dl_train, dl_val, dl_test, ds_train

dl_train, dl_val, dl_test, ds_train = make_qdf_dataloaders()

with open(DIR_DATASET / f'orbitaldict_{BASIS_SET}.pickle', 'rb') as f:
    ORBITAL_DICT = pickle.load(f)
N_ORBITALS = len(ORBITAL_DICT)

sample    = ds_train[0]
N_OUTPUT  = int(np.asarray(sample[6]).shape[1]) if len(sample) >= 8 else 1
print(f'train={len(ds_train)} / val/test datasets ready')
print(f'N_orbitals={N_ORBITALS}, N_output={N_OUTPUT}')
print(f'orbital_dict path: {DIR_DATASET}/orbitaldict_{BASIS_SET}.pickle')

In [ ]:
# ── Trainer 정의 ───────────────────────────────────────────────────────

class EOnlyTrainer:
    """에너지(E) 손실만으로 학습. V (HK map) 태스크 비활성화."""
    def __init__(self, model, lr, lr_decay, step_size):
        self.model     = model
        self.optimizer = optim.Adam(model.parameters(), lr)
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size, lr_decay)

    def train(self, dataloader):
        losses_E = 0.0
        for data in dataloader:
            loss_E = self.model.forward(data, train=True, target='E')
            self.optimizer.zero_grad()
            loss_E.backward()
            self.optimizer.step()
            losses_E += loss_E.item()
            del loss_E, data
        self.scheduler.step()
        return losses_E, 0.0


class EVTrainer:
    """원본과 동일: E + V 이중 태스크 학습."""
    def __init__(self, model, lr, lr_decay, step_size):
        self.model     = model
        self.optimizer = optim.Adam(model.parameters(), lr)
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size, lr_decay)

    def train(self, dataloader):
        losses_E, losses_V = 0.0, 0.0
        for data in dataloader:
            loss_E = self.model.forward(data, train=True, target='E')
            self.optimizer.zero_grad()
            loss_E.backward()
            self.optimizer.step()
            losses_E += loss_E.item()

            loss_V = self.model.forward(data, train=True, target='V')
            self.optimizer.zero_grad()
            loss_V.backward()
            self.optimizer.step()
            losses_V += loss_V.item()
            del loss_E, loss_V, data
        self.scheduler.step()
        return losses_E, losses_V


def make_qdf_model():
    """재현 가능한 초기 가중치로 새 QDF 모델 반환."""
    torch.manual_seed(SEED)
    return qdf_train.QuantumDeepField(
        DEVICE, N_ORBITALS,
        QDF_DIM, QDF_LAYER_FUNCTIONAL, QDF_OPERATION, N_OUTPUT,
        QDF_HIDDEN_HK, QDF_LAYER_HK,
    ).to(DEVICE)


def run_qdf_experiment(trainer_cls, label, save_path, n_epochs=QDF_N_EPOCHS):
    """학습 후 best checkpoint를 save_path에 저장."""
    model   = make_qdf_model()
    trainer = trainer_cls(model, QDF_LR, QDF_LR_DECAY, QDF_STEP_SIZE)
    tester  = qdf_train.Tester(model)

    records = []
    best_val = float('inf')
    best_state = None
    t0 = time.time()

    print(f"\n{'='*60}")
    print(f"  실험: {label}  ({n_epochs} epochs)")
    print(f"  Epoch  Loss_E     Loss_V     MAE_val (eV)   MAE_test (eV)")
    print(f"  {'-'*58}")

    for epoch in range(n_epochs):
        loss_E, loss_V = trainer.train(dl_train)
        mae_val  = tester.test(dl_val)[0]
        mae_test = tester.test(dl_test)[0]

        mae_val_f  = float(mae_val.split(',')[0])  if ',' in mae_val  else float(mae_val)
        mae_test_f = float(mae_test.split(',')[0]) if ',' in mae_test else float(mae_test)

        if mae_val_f < best_val:
            best_val   = mae_val_f
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        records.append(dict(epoch=epoch+1, loss_E=loss_E, loss_V=loss_V,
                            mae_val=mae_val_f, mae_test=mae_test_f))

        if (epoch + 1) % max(1, n_epochs // 5) == 0 or epoch == 0:
            print(f"  {epoch+1:5d}  {loss_E:9.2f}  {loss_V:9.2f}  "
                  f"{mae_val_f:12.4f}    {mae_test_f:12.4f}")

    torch.save(best_state, save_path)
    elapsed = time.time() - t0
    print(f"  {'-'*58}")
    print(f"  Best val MAE: {best_val:.4f} eV  |  "
          f"Final test MAE: {mae_test_f:.4f} eV  |  {elapsed:.1f}s")
    print(f"  체크포인트 저장: {save_path.name}")
    del model
    return records, best_val, mae_test_f


print('Trainer 정의 완료.')

In [ ]:
# ── 실험 A: E-only ─────────────────────────────────────────────────────
rec_eonly, best_eonly, test_eonly = run_qdf_experiment(
    EOnlyTrainer, 'QDF E-only (HK map 없음)', CKPT_EONLY
)

In [ ]:
# ── 실험 B: E+V 이중 태스크 ────────────────────────────────────────────
rec_ev, best_ev, test_ev = run_qdf_experiment(
    EVTrainer, 'QDF E+V 이중 태스크 (HK map 포함)', CKPT_EV
)

In [ ]:
# ── QDF 결과 비교 ──────────────────────────────────────────────────────
import pandas as pd

df_qdf = pd.DataFrame([
    {'조건': 'E-only',      'Best val MAE (eV)': f'{best_eonly:.4f}', 'Final test MAE (eV)': f'{test_eonly:.4f}'},
    {'조건': 'E+V (이중 태스크)', 'Best val MAE (eV)': f'{best_ev:.4f}',    'Final test MAE (eV)': f'{test_ev:.4f}'},
])
print('\n[QDF QM9 정확도 비교]')
print(df_qdf.to_string(index=False))
delta = test_eonly - test_ev
print(f'\n► E+V vs E-only test MAE 차이: {delta:+.4f} eV')
print('  (양수 = E+V가 더 정확)')

---
## 2. QDF → ESOL MMFF 컨포머 추론

두 QDF 체크포인트(E-only, E+V)로 ESOL 데이터셋의 분자별 4개 MMFF 컨포머 에너지를 예측합니다.  
예측값은 3DGCL 사전학습의 **auxiliary supervision target**으로 사용됩니다.

In [ ]:
# ── QDF 추론 유틸리티 (qdf_mmff_predict.py 로직 인라인) ────────────────
import preprocess as qdf_preprocess
from dig.threedgraph.dataset import MoleculeNet

SLOT_POS_ATTRS = {1: 'max1pos_mmff', 2: 'max2pos_mmff',
                  3: 'max3pos_mmff', 4: 'max4pos_mmff'}


def _parse_basis(basis_set: str):
    digits = basis_set[:-1].replace('-', '')
    nums   = [int(b) for b in digits]
    return nums[0], sum(nums[1:])


def _build_conformer_record(idx_str, z, pos, sphere, orbital_dict, inner, outer):
    """6-element QDF input record for one (atoms, coords) pair. None = skip."""
    z_list = z.detach().cpu().tolist()
    p = pos.detach().cpu().to(torch.float64).numpy()
    if not np.isfinite(p).all():
        return None
    if p.shape[0] != len(z_list) or p.shape[1] != 3:
        return None

    all_atoms = qdf_preprocess.all_atoms
    atomic_numbers, atomic_coords = [], []
    atomic_orbitals, orbital_coords, quantum_numbers = [], [], []
    n_electrons = 0

    for an, xyz in zip(z_list, p.tolist()):
        an = int(an)
        if not 1 <= an <= len(all_atoms):
            return None
        atom = all_atoms[an - 1]
        atomic_numbers.append([an])
        n_electrons += an
        atomic_coords.append(xyz)

        if an <= 2:
            aqs = [(atom + '1s' + str(i), 1) for i in range(outer)]
        else:
            aqs = (
                [(atom + '1s' + str(i), 1) for i in range(inner)]
                + [(atom + '2s' + str(i), 2) for i in range(outer)]
                + [(atom + '2p' + str(i), 2) for i in range(outer)]
            )
        for o_key, q in aqs:
            o_idx = orbital_dict.get(o_key)
            if o_idx is None:
                return None
            atomic_orbitals.append(int(o_idx))
            orbital_coords.append(xyz)
            quantum_numbers.append(q)

    atomic_coords_np  = np.asarray(atomic_coords,  dtype=np.float64)
    orbital_coords_np = np.asarray(orbital_coords, dtype=np.float64)
    atomic_numbers_np = np.asarray(atomic_numbers, dtype=np.int64)
    atomic_orbitals_np  = np.asarray(atomic_orbitals,  dtype=np.int64)
    quantum_numbers_np  = np.asarray([quantum_numbers], dtype=np.float32)
    n_electrons_np      = np.asarray([[n_electrons]],   dtype=np.float32)

    field_coords = qdf_preprocess.create_field(sphere, atomic_coords_np)
    dm_orb  = qdf_preprocess.create_distancematrix(field_coords, orbital_coords_np)
    dm_atom = qdf_preprocess.create_distancematrix(field_coords, atomic_coords_np)
    potential = qdf_preprocess.create_potential(dm_atom, atomic_numbers_np)
    n_field   = int(field_coords.shape[0])

    return [
        idx_str,
        atomic_orbitals_np,
        dm_orb.astype(np.float32),
        quantum_numbers_np.astype(np.float32),
        n_electrons_np.astype(np.float32),
        n_field,
        np.zeros((1, 2), dtype=np.float32),
        potential.astype(np.float32).reshape(n_field, 1),
    ]


def _predict_batch(model, records):
    """QDF 배치 추론. [B, N_output] numpy 반환."""
    data = list(zip(*records))
    with torch.no_grad():
        _ids, E_ = model.forward(tuple(data), predict=True)
    return E_.detach().cpu().numpy()


def run_esol_inference(
    ckpt_path: Path,
    out_csv: Path,
    label: str = '',
    batch_size: int = 8,
    limit: int = 0,
):
    """
    QDF 체크포인트로 ESOL MMFF 컨포머 에너지 예측.
    out_csv 경로에 smiles,slot,energy CSV 저장.
    """
    inner, outer = _parse_basis(BASIS_SET)
    sphere = qdf_preprocess.create_sphere(float(RADIUS), float(GRID_INTERVAL))

    model = qdf_train.QuantumDeepField(
        DEVICE, N_ORBITALS,
        QDF_DIM, QDF_LAYER_FUNCTIONAL, QDF_OPERATION, N_OUTPUT,
        QDF_HIDDEN_HK, QDF_LAYER_HK,
    ).to(DEVICE)
    state = torch.load(str(ckpt_path), map_location=DEVICE, weights_only=False)
    model.load_state_dict(state)
    model.eval()

    ds = MoleculeNet(root=str(REPO_ROOT / 'dataset'), name='esol')
    N  = len(ds) if not limit else min(limit, len(ds))

    batch_meta, batch_records = [], []
    n_rows = n_skip = 0
    t0 = time.perf_counter()

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with out_csv.open('w', newline='', encoding='utf-8') as fh:
        writer = csv.writer(fh)
        writer.writerow(['smiles', 'slot', 'energy'])

        def _flush():
            nonlocal n_rows
            if not batch_records:
                return
            E = _predict_batch(model, batch_records)
            for (smi, slot), e in zip(batch_meta, E):
                writer.writerow([smi, slot, float(e[0])])
                n_rows += 1
            batch_meta.clear()
            batch_records.clear()

        for i in range(N):
            data = ds[i]
            smi = getattr(data, 'smiles', None)
            z   = getattr(data, 'z', None)
            if smi is None or z is None:
                n_skip += 4; continue

            for slot in (1, 2, 3, 4):
                pos = getattr(data, SLOT_POS_ATTRS[slot], None)
                if pos is None or pos.numel() == 0:
                    n_skip += 1; continue
                rec = _build_conformer_record(
                    f'{i}_{slot}', z, pos, sphere, ORBITAL_DICT, inner, outer
                )
                if rec is None:
                    n_skip += 1; continue
                batch_meta.append((str(smi), slot))
                batch_records.append(rec)
                if len(batch_records) >= batch_size:
                    _flush()

        _flush()

    del model
    gc.collect()
    elapsed = time.perf_counter() - t0
    print(f'[{label}] rows={n_rows}  skipped={n_skip}  elapsed={elapsed:.1f}s  -> {out_csv.name}')
    return n_rows


print('추론 유틸리티 정의 완료.')

In [ ]:
# ── E-only QDF로 ESOL 추론 ─────────────────────────────────────────────
n_eonly = run_esol_inference(CKPT_EONLY, CSV_EONLY, label='E-only')

In [ ]:
# ── E+V QDF로 ESOL 추론 ────────────────────────────────────────────────
n_ev = run_esol_inference(CKPT_EV, CSV_EV, label='E+V')

In [ ]:
# ── CSV → aux .pt 변환 ─────────────────────────────────────────────────

def build_aux_pt(csv_path: Path, out_pt: Path, label: str = '') -> dict:
    """
    smiles,slot,energy CSV → {smiles: mean_energy_across_4slots}.
    {"targets": {...}, "meta": {...}} 형식으로 .pt 저장.
    """
    # slot별 에너지 수집
    smiles_slots: dict[str, list[float]] = {}
    with csv_path.open('r', newline='', encoding='utf-8') as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            smi = row['smiles']
            e   = float(row['energy'])
            smiles_slots.setdefault(smi, []).append(e)

    # 슬롯 평균 (aux target = 4 컨포머 에너지의 평균)
    targets = {smi: float(np.mean(vals)) for smi, vals in smiles_slots.items()}

    meta = {
        'qdf_property': 'atomization',
        'scalar': 'energy',
        'n_molecules': len(targets),
        'label': label,
        'pred_csv': str(csv_path),
    }
    torch.save({'targets': targets, 'meta': meta}, str(out_pt))

    e_vals = np.array(list(targets.values()))
    print(f'[{label}] aux.pt: {len(targets)} 분자  '
          f'mean={e_vals.mean():.2f} eV  std={e_vals.std():.2f} eV  -> {out_pt.name}')
    return targets


targets_eonly = build_aux_pt(CSV_EONLY, AUX_EONLY, label='E-only')
targets_ev    = build_aux_pt(CSV_EV,    AUX_EV,    label='E+V')

# 두 모델의 예측 분포 비교
smi_common = set(targets_eonly) & set(targets_ev)
diff = np.array([targets_ev[s] - targets_eonly[s] for s in smi_common])
print(f'\n공통 분자 수: {len(smi_common)}')
print(f'E+V - E-only 에너지 차이:  mean={diff.mean():.4f} eV  std={diff.std():.4f} eV')

In [ ]:
# ── 두 QDF 모델 예측값 분포 시각화 ─────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

smis = sorted(smi_common)
vals_eonly = np.array([targets_eonly[s] for s in smis])
vals_ev    = np.array([targets_ev[s]    for s in smis])

# 히스토그램 비교
axes[0].hist(vals_eonly, bins=30, alpha=0.6, label='E-only QDF', color='royalblue')
axes[0].hist(vals_ev,    bins=30, alpha=0.6, label='E+V QDF',   color='tomato')
axes[0].set_xlabel('Predicted Atomization Energy (eV)')
axes[0].set_ylabel('Count')
axes[0].set_title('ESOL 분자 에너지 예측 분포')
axes[0].legend()

# 산점도
axes[1].scatter(vals_eonly, vals_ev, s=8, alpha=0.5, color='purple')
mn, mx = min(vals_eonly.min(), vals_ev.min()), max(vals_eonly.max(), vals_ev.max())
axes[1].plot([mn, mx], [mn, mx], 'k--', lw=1, label='y=x')
axes[1].set_xlabel('E-only QDF 예측 (eV)')
axes[1].set_ylabel('E+V QDF 예측 (eV)')
axes[1].set_title('E-only vs E+V 예측값 비교')
axes[1].legend()

plt.tight_layout()
fig.savefig(NB_DIR / 'ensemble_aux_compare.png', dpi=130, bbox_inches='tight')
plt.show()
print('그래프 저장: ensemble_aux_compare.png')

---
## 3. 3DGCL 사전학습 + 파인튜닝 비교

세 가지 조건으로 GraphCL을 사전학습하고 ESOL 파인튜닝 RMSE를 비교합니다.

| 조건 | aug | QDF aux λ | aux 타겟 |
|------|-----|-----------|----------|
| **A** 기준선 | MMFFrandom | 0.0 | 없음 |
| **B** E-only | MMFFweighted | 0.05 | E-only QDF 예측 |
| **C** E+V    | MMFFweighted | 0.05 | E+V QDF 예측 |

> 컨포머 선택 가중치(`weights_path`)는 B/C 모두 동일한 파일 사용.
> aux 타겟의 품질 차이(E-only vs E+V)가 3DGCL 정확도에 미치는 영향을 격리합니다.

In [ ]:
# ── pretrain_quality_core import ──────────────────────────────────────
import pretrain_quality_core as q
from dig.sslgraph.utils.device import pick_torch_device

DGCL_DEVICE = pick_torch_device()
DGCL_ROOT   = REPO_ROOT / 'dataset'
MODEL_ROOT  = REPO_ROOT / 'models' / 'ensemble_comparison'
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

print(f'DGCL device : {DGCL_DEVICE}')
print(f'weights_pt  : {WEIGHTS_PT.name}  ({"OK" if WEIGHTS_PT.exists() else "MISSING"})')
print(f'aux_eonly   : {AUX_EONLY.name}   ({"OK" if AUX_EONLY.exists() else "MISSING"})')
print(f'aux_ev      : {AUX_EV.name}      ({"OK" if AUX_EV.exists() else "MISSING"})')
print(f'model_root  : {MODEL_ROOT}')

In [ ]:
# ── Side A: 기준선 (no QDF) ────────────────────────────────────────────
print('\n>>> Side A: GraphCL 기준선 (QDF aux 없음)')

summary_a = q.run_pretrain_side(
    'A',
    dataset=DGCL_DATASET,
    batch_size=DGCL_BATCH,
    p_epoch=P_EPOCH,
    model_root=MODEL_ROOT / 'A',
    device=DGCL_DEVICE,
    seed=SEED,
    dataset_root=DGCL_ROOT,
)
print(f'A pretrain done: {summary_a.get("best_ckpt")}')

In [ ]:
# ── Side B: E-only QDF aux ────────────────────────────────────────────
print('\n>>> Side B: GraphCL + E-only QDF aux')

if not WEIGHTS_PT.exists():
    raise FileNotFoundError(
        f'weights_path not found: {WEIGHTS_PT}\n'
        'Run: python examples/sslgraph/bench/compute_mmff_weights.py'
    )

summary_b = q.run_pretrain_side(
    'B',
    dataset=DGCL_DATASET,
    batch_size=DGCL_BATCH,
    p_epoch=P_EPOCH,
    model_root=MODEL_ROOT / 'B',
    device=DGCL_DEVICE,
    weights_path=WEIGHTS_PT,
    weight_kT=WEIGHT_KT,
    weight_norm='zscore',
    seed=SEED,
    dataset_root=DGCL_ROOT,
    qdf_aux_lambda=QDF_AUX_LAMBDA,
    qdf_aux_pt=AUX_EONLY,
)
print(f'B pretrain done: {summary_b.get("best_ckpt")}')

In [ ]:
# ── Side C: E+V QDF aux ───────────────────────────────────────────────
print('\n>>> Side C: GraphCL + E+V QDF aux')

summary_c = q.run_pretrain_side(
    'C',
    dataset=DGCL_DATASET,
    batch_size=DGCL_BATCH,
    p_epoch=P_EPOCH,
    model_root=MODEL_ROOT / 'C',
    device=DGCL_DEVICE,
    weights_path=WEIGHTS_PT,
    weight_kT=WEIGHT_KT,
    weight_norm='zscore',
    seed=SEED,
    dataset_root=DGCL_ROOT,
    qdf_aux_lambda=QDF_AUX_LAMBDA,
    qdf_aux_pt=AUX_EV,
)
print(f'C pretrain done: {summary_c.get("best_ckpt")}')

In [ ]:
# ── 파인튜닝: A / B / C ───────────────────────────────────────────────
common_ft = dict(
    dataset=DGCL_DATASET,
    batch_size=DGCL_BATCH,
    f_epoch=F_EPOCH,
    n_times=N_TIMES,
    n_folds=N_FOLDS,
    device=DGCL_DEVICE,
    seed=SEED,
    dataset_root=DGCL_ROOT,
)

print('\n>>> 파인튜닝 A ...')
ft_a = q.run_finetune_side(summary_a, **common_ft)
print(f'  A RMSE: {ft_a["rmse_mean"]:.4f} ± {ft_a["rmse_sd"]:.4f}')

print('>>> 파인튜닝 B ...')
ft_b = q.run_finetune_side(summary_b, **common_ft)
print(f'  B RMSE: {ft_b["rmse_mean"]:.4f} ± {ft_b["rmse_sd"]:.4f}')

print('>>> 파인튜닝 C ...')
ft_c = q.run_finetune_side(summary_c, **common_ft)
print(f'  C RMSE: {ft_c["rmse_mean"]:.4f} ± {ft_c["rmse_sd"]:.4f}')

---
## 4. 최종 결과 비교

In [ ]:
# ── 최종 결과 표 ───────────────────────────────────────────────────────
rows = [
    {
        '조건':         'A: 기준선 (no QDF)',
        'QDF aux':     '없음',
        'QDF test MAE (eV)': '-',
        'ESOL RMSE':   f'{ft_a["rmse_mean"]:.4f} ± {ft_a["rmse_sd"]:.4f}',
        '개선율':       '기준',
    },
    {
        '조건':         'B: E-only QDF',
        'QDF aux':     'E-only 예측',
        'QDF test MAE (eV)': f'{test_eonly:.4f}',
        'ESOL RMSE':   f'{ft_b["rmse_mean"]:.4f} ± {ft_b["rmse_sd"]:.4f}',
        '개선율':       f'{(ft_a["rmse_mean"] - ft_b["rmse_mean"]) / ft_a["rmse_mean"] * 100:+.1f}%',
    },
    {
        '조건':         'C: E+V QDF (이중 태스크)',
        'QDF aux':     'E+V 예측',
        'QDF test MAE (eV)': f'{test_ev:.4f}',
        'ESOL RMSE':   f'{ft_c["rmse_mean"]:.4f} ± {ft_c["rmse_sd"]:.4f}',
        '개선율':       f'{(ft_a["rmse_mean"] - ft_c["rmse_mean"]) / ft_a["rmse_mean"] * 100:+.1f}%',
    },
]

df_final = pd.DataFrame(rows)
print('\n' + '='*70)
print(f'  최종 결과 (P_EPOCH={P_EPOCH}, F_EPOCH={F_EPOCH}, BUDGET={QUALITY_BUDGET})')
print('='*70)
print(df_final.to_string(index=False))
print()

# 핵심 비교
ev_vs_a  = ft_a['rmse_mean'] - ft_c['rmse_mean']
ev_vs_b  = ft_b['rmse_mean'] - ft_c['rmse_mean']
print(f'► C(E+V) vs A(baseline)  RMSE 차이: {ev_vs_a:+.4f}  ({ev_vs_a/ft_a["rmse_mean"]*100:+.1f}%)')
print(f'► C(E+V) vs B(E-only)    RMSE 차이: {ev_vs_b:+.4f}  ({ev_vs_b/ft_b["rmse_mean"]*100:+.1f}%)')

if ev_vs_b > 0:
    print('\n✓ E+V QDF가 E-only보다 더 좋은 aux 신호를 제공하여 ESOL RMSE가 낮습니다.')
else:
    print('\n※ 현재 epoch 수에서는 차이가 미미합니다.')
    print('  QDF를 더 오래 학습하거나(QDF_N_EPOCHS↑), 3DGCL budget을 늘리면(two_hour) 차이가 드러납니다.')

In [ ]:
# ── 결과 시각화 ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── 왼쪽: QDF QM9 정확도 비교 ────────────────────────────────────────
ax = axes[0]
epochs_eonly = [r['epoch'] for r in rec_eonly]
epochs_ev    = [r['epoch'] for r in rec_ev]
ax.plot(epochs_eonly, [r['mae_val']  for r in rec_eonly], 'b--', label='E-only val',  lw=1.5)
ax.plot(epochs_eonly, [r['mae_test'] for r in rec_eonly], 'b-',  label='E-only test', lw=1.5)
ax.plot(epochs_ev,    [r['mae_val']  for r in rec_ev],    'r--', label='E+V val',     lw=1.5)
ax.plot(epochs_ev,    [r['mae_test'] for r in rec_ev],    'r-',  label='E+V test',    lw=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE (eV)')
ax.set_title('Step 1: QDF Learning Curve\n(QM9under7atoms, atomization eV)')
ax.legend(fontsize=8)

# ── 가운데: ESOL aux 에너지 예측 품질 ────────────────────────────────
ax = axes[1]
smis_sorted = sorted(smi_common)
v_e = np.array([targets_eonly[s] for s in smis_sorted])
v_v = np.array([targets_ev[s]    for s in smis_sorted])
ax.scatter(v_e, v_v, s=6, alpha=0.4, color='purple', label='분자별 예측')
mn2, mx2 = min(v_e.min(), v_v.min()), max(v_e.max(), v_v.max())
ax.plot([mn2, mx2], [mn2, mx2], 'k--', lw=1)
ax.set_xlabel('E-only QDF 예측 (eV)')
ax.set_ylabel('E+V QDF 예측 (eV)')
ax.set_title('Step 2: ESOL aux 예측값 비교\n(E-only vs E+V)')
ax.legend(fontsize=8)

# ── 오른쪽: 3DGCL ESOL RMSE 비교 ────────────────────────────────────
ax = axes[2]
labels = ['A\n(no QDF)', 'B\n(E-only)', 'C\n(E+V)']
rmse_m = [ft_a['rmse_mean'], ft_b['rmse_mean'], ft_c['rmse_mean']]
rmse_s = [ft_a['rmse_sd'],   ft_b['rmse_sd'],   ft_c['rmse_sd']]
colors = ['#888888', 'royalblue', 'tomato']
bars = ax.bar(labels, rmse_m, yerr=rmse_s, capsize=6, color=colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, rmse_m):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(rmse_s) * 0.1,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('ESOL RMSE (log mol/L)')
ax.set_title('Step 3: 3DGCL ESOL 파인튜닝 RMSE\n(낮을수록 좋음)')
ax.set_ylim(0, max(rmse_m) * 1.3)

plt.tight_layout()
fig.savefig(NB_DIR / 'ensemble_final_compare.png', dpi=150, bbox_inches='tight')
plt.show()
print('그래프 저장: ensemble_final_compare.png')

In [ ]:
# ── 결과 JSON 저장 ────────────────────────────────────────────────────
import json

result_json = {
    'config': {
        'qdf_dataset':    QDF_DATASET,
        'qdf_n_epochs':   QDF_N_EPOCHS,
        'qdf_dim':        QDF_DIM,
        'quality_budget': QUALITY_BUDGET,
        'p_epoch':        P_EPOCH,
        'f_epoch':        F_EPOCH,
        'n_folds':        N_FOLDS,
        'qdf_aux_lambda': QDF_AUX_LAMBDA,
        'seed':           SEED,
    },
    'qdf': {
        'eonly_best_val_mae': best_eonly,
        'eonly_final_test_mae': test_eonly,
        'ev_best_val_mae':   best_ev,
        'ev_final_test_mae': test_ev,
    },
    'esol_finetune': {
        'A_no_aux':  {'rmse_mean': ft_a['rmse_mean'], 'rmse_sd': ft_a['rmse_sd']},
        'B_eonly':   {'rmse_mean': ft_b['rmse_mean'], 'rmse_sd': ft_b['rmse_sd']},
        'C_ev':      {'rmse_mean': ft_c['rmse_mean'], 'rmse_sd': ft_c['rmse_sd']},
    },
}

result_path = NB_DIR / 'ensemble_comparison_results.json'
with result_path.open('w') as f:
    json.dump(result_json, f, indent=2)

print(f'결과 저장: {result_path.name}')
print(json.dumps(result_json, indent=2))

---
## 해석 가이드

### QDF 품질이 3DGCL에 미치는 경로

```
E+V 이중 태스크 학습
  → HK map 보조 손실이 QDF 인코더를 물리적으로 정규화
  → 더 정확한 원자화 에너지 예측 (test MAE ↓)
  → ESOL MMFF 컨포머에 대한 더 신뢰할 수 있는 에너지 타겟
  → 3DGCL 사전학습 중 aux MSE 신호의 품질 향상
  → 인코더가 에너지에 민감한 분자 표현 학습
  → ESOL 파인튜닝 RMSE 감소
```

### 더 좋은 결과를 위한 설정

| 파라미터 | 현재 값 | 권장 값 | 예상 효과 |
|---------|---------|---------|----------|
| `QDF_N_EPOCHS` | 15 | 100+ | QDF test MAE ↓↓ |
| `QDF_DIM` | 50 | 200 | 원본 논문 설정 |
| `QDF_DATASET` | under7atoms | under14atoms | 더 다양한 분자 커버 |
| `QUALITY_BUDGET` | smoke | two_hour | 3DGCL 정확도 ↑ |
| `QDF_AUX_LAMBDA` | 0.05 | 0.05~0.1 | 튜닝 필요 |

### 사전 학습된 QDF 사용 (권장)

```python
# 이미 학습된 E+V QDF 예측 파일 사용:
AUX_EV_PRETRAINED = REPO_ROOT / 'dataset' / 'esol_qdf_aux_atomization.pt'
```

이 파일은 QM9under14atoms에서 2000 epoch 학습한 E+V QDF로 생성됩니다.  
위 실험의 E+V aux(15 epoch) 대신 이 파일을 `AUX_EV`에 할당하면 더 큰 개선 효과를 볼 수 있습니다.